In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

df = pd.read_csv("data/sales.csv", parse_dates=["date"])
df["revenue"] = df["qty"] * df["price"]
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
df.head()

In [ ]:
print(df.shape)
print("Пропуски, %:")
print(df.isna().mean().mul(100).round(2))
df[["qty", "price", "cost"]].describe()

In [ ]:
total = df.groupby("month", as_index=False)["revenue"].sum()
by_cat = df.groupby(["month", "category"], as_index=False)["revenue"].sum()

px.line(total, x="month", y="revenue",
        title="Выручка по месяцам (всего)").show()
px.line(by_cat, x="month", y="revenue", color="category",
        title="Выручка по месяцам по категориям").show()

In [ ]:
limit = df.groupby("product")["price"].transform(lambda s: s.quantile(0.99))
df["price_clean"] = df["price"].where(df["price"] <= limit)

avg_price = df.groupby(["month", "category"], as_index=False)["price_clean"].mean()

plt.figure(figsize=(11, 5))
sns.lineplot(data=avg_price, x="month", y="price_clean",
             hue="category", marker="o")
plt.title("Средняя цена по категориям")
plt.xlabel("Месяц")
plt.ylabel("Средняя цена, руб.")
plt.show()

In [ ]:
by_store = df.groupby(["store_id", "city"], as_index=False)["qty"].sum()
by_store["label"] = by_store["store_id"] + "\n" + by_store["city"]
by_city = df.groupby("city")["revenue"].sum().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(by_store["label"], by_store["qty"], color="tab:blue")
axes[0].set_title("Проданное количество по магазинам")
axes[0].set_ylabel("Единиц товара")

axes[1].barh(by_city.index, by_city.values, color="tab:orange")
axes[1].set_title("Выручка по городам")
axes[1].set_xlabel("Руб.")
fig.tight_layout()
plt.show()

In [ ]:
days = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
df["weekday"] = df["date"].dt.dayofweek.map(dict(enumerate(days)))

plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="weekday", y="qty", order=days, errorbar=None)
plt.title("Среднее количество в записи по дням недели")
plt.xlabel("День недели")
plt.ylabel("Среднее qty")
plt.show()

In [ ]:
pay = df["payment_method"].value_counts().reset_index()
pay.columns = ["payment_method", "count"]
px.pie(pay, names="payment_method", values="count",
       title="Структура способов оплаты").show()

pay_year = (df.assign(year=df["date"].dt.year)
              .groupby(["year", "payment_method"]).size().reset_index(name="count"))
px.bar(pay_year, x="year", y="count", color="payment_method", barmode="relative",
       title="Способы оплаты по годам").show()

In [ ]:
d = df.sort_values(["store_id", "product", "date"]).copy()
g = d.groupby(["store_id", "product"])["promo"]

prev3 = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).max()).fillna(0)
next3 = g.transform(
    lambda s: s[::-1].shift(1).rolling(3, min_periods=1).max()[::-1]).fillna(0)

d["phase"] = np.select(
    [d["promo"] == 1, prev3 == 1, next3 == 1],
    ["во время", "после", "до"], default="обычные дни")

d["rel_qty"] = (d["qty"] / d.groupby(["store_id", "product"])["qty"]
                .transform("mean")).clip(upper=3)

order = ["обычные дни", "до", "во время", "после"]
res = d.groupby("phase")["rel_qty"].mean().reindex(order)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(res.index, res.values, color=["gray", "tab:blue", "tab:green", "tab:red"])
ax.axhline(1, color="black", linestyle="--", linewidth=1)
ax.set_title("Относительный спрос до / во время / после акции")
ax.set_ylabel("qty / средний qty товара в магазине")
plt.show()

print(d.groupby("promo")["price_clean"].mean() if "price_clean" in d else "")